### MindSpore NLP 机器翻译 (Helsinki-NLP / T5)

本案例将演示如何使用 MindSpore NLP 在昇腾 (Ascend) 环境下微调一个机器翻译模型。我们将使用 WMT16 数据集将英语翻译为罗马尼亚语。

本案例的运行环境为：

| Python | MindSpore | MindSpore NLP |
| :----- | :-------- | :------------ |
| 3.10   | 2.7.0     | 0.5.1  |

In [ ]:
#若在https://internstudio-ascend.intern-ai.org.cn/console/instance进行开发时，使用notebook会出现无法正常使用NPU，可进行以下步骤：
# 进入开发机的命令窗口
# 1. 激活你的环境
# conda activate mind_py310
# pip install ipykernel
# python -m ipykernel install --user --name=mind_py310 --display-name="Python (mind_py310)"
# 2. 加载系统基础驱动配置
# source /usr/local/Ascend/ascend-toolkit/set_env.sh
# 3.【核心步骤】手动补全深层驱动路径 (修复 libascend_hal.so 报错)
# export LD_LIBRARY_PATH=/usr/local/Ascend/driver/lib64/driver:/usr/local/Ascend/driver/lib64/common:/usr/local/Ascend/driver/lib64:$LD_LIBRARY_PATH
# 4. 启动 Jupyter Lab
#jupyter lab --allow-root

In [ ]:
# 安装依赖
# !pip install mindnlp==0.5.1
# !pip install sacrebleu

#### Step 1: 兼容性修复与环境配置
首先，我们需要应用一个补丁来修复 mindtorch 的版本兼容性问题，并配置 MindSpore 的运行环境。

In [ ]:
# --------兼容性补丁---------
import mindtorch.autograd.function
if not hasattr(mindtorch.autograd.function, 'FunctionCtx'):
    class FunctionCtx:
        def __init__(self):
            self.saved_tensors = ()
        def save_for_backward(self, *tensors):
            self.saved_tensors = tensors
    mindtorch.autograd.function.FunctionCtx = FunctionCtx
    print("已应用 mindtorch 兼容性修复")

# ----基础配置与环境 ----
import mindnlp
import mindspore
from mindspore import context
from datasets import load_dataset
import evaluate
import numpy as np
import os

# 1. 设置 Token (请确保这是有效的 HuggingFace Token)
os.environ["HF_TOKEN"] = "hf_***" 

# 2. 清理离线环境变量，确保联网 (如果需要在线下载)
if 'HF_DATASETS_OFFLINE' in os.environ: del os.environ['HF_DATASETS_OFFLINE']
if 'TRANSFORMERS_OFFLINE' in os.environ: del os.environ['TRANSFORMERS_OFFLINE']
mindspore.set_seed(42)
print("环境配置完成")

#### Step 2: 模型选择
支持 T5 系列模型并添加相应的前缀，也支持 Helsinki-NLP 专用翻译模型

In [ ]:
model_checkpoint = "Helsinki-NLP/opus-mt-en-ro"
# model_checkpoint = "t5-small"
print(f"当前选用的模型: {model_checkpoint}")

#### Step 3: 数据加载与预处理
本案例使用 WMT 数据集，这是一个汇集了包括新闻评论和议会记录在内的多种来源的机器翻译数据集。
我们将展示如何加载用于此任务的数据集，以及如何使用 MindSpore NLP 提供的 Trainer 接口（类似 Hugging Face 的体验）在 Ascend NPU 环境下对模型进行高效微调。
接着，我们初始化 Tokenizer 并进行针对性的模型设置：
1. **T5 模型特殊处理**：由于 T5 是一个多任务模型（Text-to-Text），我们需要给输入文本添加特定的前缀（如 `"translate English to Romanian: "`）来告诉模型执行翻译任务。
2. **mBART 模型特殊处理**：如果是 mBART 模型，需要显式指定源语言和目标语言的代码。

最后，我们定义了 `preprocess_function` 对数据进行批量处理：
- 将输入文本（Input）和目标文本（Label）分别进行 Tokenization。
- 使用 `map` 函数将预处理应用到整个数据集。
- **注意**：为了快速演示流程，这里只采样了少量数据（2000条训练数据）。正式训练时请注释掉 `.select()` 部分以使用全量数据。

In [ ]:
raw_datasets = load_dataset("wmt16", "ro-en")
metric = evaluate.load("sacrebleu")
print("数据集加载完成")

from mindnlp.transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# --- 针对 mBART 的特殊处理 ---
if "mbart" in model_checkpoint:
    print("检测到 mBART 模型，设置源语言和目标语言代码...")
    tokenizer.src_lang = "en-XX"
    tokenizer.tgt_lang = "ro-RO"

# --- 针对 T5 的前缀判断 ---
if model_checkpoint in ["t5-small", "t5-base", "t5-large", "t5-3b", "t5-11b"]:
    prefix = "translate English to Romanian: "
    print(f"检测到 T5 模型，已添加前缀: '{prefix}'")
else:
    prefix = ""
    print("非 T5 模型，不添加前缀。")


# --- 数据预处理函数 ---
max_input_length = 128
max_target_length = 128
source_lang = "en"
target_lang = "ro"

def preprocess_function(examples):
    # inputs: 前缀 + 原文
    inputs = [prefix + ex[source_lang] for ex in examples["translation"]]
    targets = [ex[target_lang] for ex in examples["translation"]]
    
    model_inputs = tokenizer(inputs, max_length=max_input_length, truncation=True)

    # targets 需要用 tokenizer 处理为 labels
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(targets, max_length=max_target_length, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 采样少量数据用于快速演示 (正式训练注释掉select)
train_dataset = raw_datasets["train"].shuffle(seed=42).select(range(2000))
val_dataset = raw_datasets["validation"].shuffle(seed=42).select(range(500))
test_dataset = raw_datasets["test"].shuffle(seed=42).select(range(200))

# 批量映射处理
print("正在处理数据 (Tokenization)...")
tokenized_datasets = {
    "train": train_dataset.map(preprocess_function, batched=True, remove_columns=raw_datasets["train"].column_names),
    "validation": val_dataset.map(preprocess_function, batched=True, remove_columns=raw_datasets["validation"].column_names),
    "test": test_dataset.map(preprocess_function, batched=True, remove_columns=raw_datasets["test"].column_names)
}
print("数据预处理完成")

#### Step 4: 加载模型

In [ ]:
from mindnlp.transformers import AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)
print("模型加载成功")

#### Step 5: 定义评估指标 (BLEU)

需要定义 `compute_metrics` 函数来评估翻译质量。该函数负责将模型输出的 ID 序列解码为文本，处理标签中的特殊掩码（-100），并最终计算 **BLEU** 分数和生成序列的平均长度。

In [ ]:
# ---- 定义评估函数 ----
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    # 将 -100 替换为 pad token 才能解码
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"bleu": result["score"]}
    
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    return {k: round(v, 4) for k, v in result.items()}

#### Step 6: 配置 Trainer 并开始训练

为了微调模型，我们需要使用 MindSpore NLP 提供的 `Seq2SeqTrainer` 接口。首先，我们通过 `Seq2SeqTrainingArguments` 定义训练的具体配置：

- **output_dir**: 模型检查点和日志的保存路径。
- **learning_rate**: 设置为 2e-5，微调通常使用较小的学习率。
- **per_device_train_batch_size**: 根据显存大小设置为 16。
- **predict_with_generate**: 设置为 `True`，以便在评估时生成翻译结果并计算指标。
- **metric_for_best_model**: 指定 "bleu" 作为评估模型好坏的指标，并保存最优模型。

定义好参数后，我们构建 `Seq2SeqTrainer` 对象并运行 `train()` 方法开始训练。

In [ ]:
from mindnlp.transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments

batch_size = 16
# 自动生成输出目录名
model_name = model_checkpoint.split("/")[-1]
output_dir = f"{model_name}-finetuned-{source_lang}-to-{target_lang}"

args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=1,
    predict_with_generate=True,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("开始训练...")
try:
    mindspore.hal.empty_cache()
except:
    pass
trainer.train()
print("训练结束")

#### Step 7: 推理与全量评估

In [ ]:
# --- 推理演示 (Inference) ----

print("\n=== 单句推理测试 ===")
src_text = "Machine learning is fascinating."
# 再次应用前缀逻辑，确保推理时格式和训练时一致
input_text = prefix + src_text 

inputs = tokenizer(input_text, return_tensors="ms", max_length=128, truncation=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

outputs = model.generate(
    input_ids=inputs["input_ids"], 
    attention_mask=inputs["attention_mask"], 
    max_new_tokens=40, 
    num_beams=4
)
print(f"输入: {input_text}")
print(f"翻译: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")


# --- 全量测试集评估 (计算 BLEU) ---
print("\n=== 正在计算测试集 BLEU 分数 ===")
# trainer.predict 会自动处理批量推理、设备分配和指标计算
test_results = trainer.predict(tokenized_datasets["test"])
print("测试集最终指标:", test_results.metrics)